# LAB 2: Search Pipeline com RRF e Min-Max
## Hybrid Search no OpenSearch 3.x com embeddings BGE-M3 via Ollama (cliente-side)

**Objetivo:** Configurar e testar **Search Pipelines** com **RRF** (sintaxe oficial OpenSearch 2.19+: `score-ranker-processor`) e **Min-Max normalization** (`normalization-processor`) para combinar BM25 + kNN sobre o corpus jurídico TCU 2026. Os embeddings da query são gerados **explicitamente via Ollama HTTP** (`/api/embeddings`, modelo `bge-m3`, dim=1024).

**Referência:** BOGAN, R. et al. *Introducing reciprocal rank fusion for hybrid search*. OpenSearch Blog, 2025. <https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/>

**Pré-requisitos:**
- **LAB1** executado — índice `corpus_juridico_aula4` populado (`lab1_config.json` disponível)
- Ollama rodando com `bge-m3` (`ollama pull bge-m3`) em `http://localhost:11434`

**Duração estimada:** 60 minutos

## 1. Configuração

In [ ]:
import subprocess, sys
for pkg in ['opensearch-py>=2.7', 'pandas', 'matplotlib', 'python-dotenv', 'requests']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import json, os, time, warnings, requests
from opensearchpy import OpenSearch
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from dotenv import load_dotenv
warnings.filterwarnings('ignore'); load_dotenv()

OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASS', 'admin')

OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
EMBED_DIM = 1024

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, ssl_show_warn=False, http_compress=True,
)
print(f"OpenSearch {client.info()['version']['number']} OK")
print(f"Ollama embeddings: {OLLAMA_BASE_URL} ({OLLAMA_EMBED_MODEL}, dim={EMBED_DIM})")

In [ ]:
with open('lab1_config.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)
INDEX_NAME = cfg['index_name']
assert cfg['embed_dim'] == EMBED_DIM, 'dim do índice diverge de BGE-M3 (1024)'
n = client.count(index=INDEX_NAME)['count']
assert n > 0, 'Execute o LAB1 primeiro.'
print(f'Índice: {INDEX_NAME} | docs indexados: {n} | dim: {EMBED_DIM}')

## 2. Função de Embedding via Ollama (cliente-side)

In [ ]:
def ollama_embed(text: str, model: str = OLLAMA_EMBED_MODEL,
                 base_url: str = OLLAMA_BASE_URL) -> list:
    """Gera embedding BGE-M3 via Ollama HTTP.
    Em caso de falha, retorna vetor zerado (1024d).
    """
    try:
        r = requests.post(
            f'{base_url}/api/embeddings',
            json={'model': model, 'prompt': text},
            timeout=60,
        )
        r.raise_for_status()
        emb = r.json().get('embedding', [])
        if len(emb) != EMBED_DIM:
            raise ValueError(f'dim {len(emb)} != {EMBED_DIM}')
        return emb
    except Exception as e:
        print(f'⚠️ Ollama indisponível ({e}). Vetor zero.')
        return [0.0] * EMBED_DIM

_v = ollama_embed('teste de embedding')
print(f'Embedding gerado pelo Ollama BGE-M3 → dim={len(_v)} | norma≈{np.linalg.norm(_v):.4f}')

## 3. Teoria: RRF e Min-Max

- **RRF (Reciprocal Rank Fusion)**: `score = 1 / (rank_constant + rank)` (default rank_constant=60). Não depende da escala dos scores originais → robusto. Processor: **`score-ranker-processor`** (OpenSearch 2.19+).
- **Min-Max Normalization**: `score_norm = (score - min) / (max - min)`. Combina via média ponderada `α·BM25 + (1-α)·kNN`. Processor: **`normalization-processor`**.

## 4. Criar Search Pipeline com RRF (`score-ranker-processor`)

In [ ]:
pipeline_rrf = 'hybrid_rrf_pipeline'
# Sintaxe oficial OpenSearch 2.19+: score-ranker-processor (Neural Search plugin).
# Ref.: https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/
rrf_body = {
    'description': 'Post processor for hybrid RRF search (BM25 + Ollama BGE-M3)',
    'phase_results_processors': [
        {
            'score-ranker-processor': {
                'combination': {
                    'technique': 'rrf',
                    'rank_constant': 60   # k da fórmula 1/(k + rank); default 60
                }
            }
        }
    ]
}
client.transport.perform_request('PUT', f'/_search/pipeline/{pipeline_rrf}', body=rrf_body)
print(f'Pipeline RRF: {pipeline_rrf} (score-ranker-processor, rank_constant=60)')

## 5. Criar Search Pipeline com Min-Max (`normalization-processor`)

In [ ]:
pipeline_minmax = 'hybrid_minmax_pipeline'
minmax_body = {
    'description': 'Pipeline hybrid com Min-Max (α=0.3 BM25 / 0.7 kNN BGE-M3)',
    'phase_results_processors': [
        {
            'normalization-processor': {
                'normalization': {'technique': 'min_max'},
                'combination':   {
                    'technique':  'arithmetic_mean',
                    'parameters': {'weights': [0.3, 0.7]}
                }
            }
        }
    ]
}
client.transport.perform_request('PUT', f'/_search/pipeline/{pipeline_minmax}', body=minmax_body)
print(f'Pipeline Min-Max: {pipeline_minmax} (normalization-processor, weights=[0.3, 0.7])')

## 6. Funções de Busca

In [ ]:
QUERY_TEXT = 'operação de crédito com garantia da União'
K = 5

def busca_bm25(q: str, k: int = K):
    body = {'size': k, 'query': {'match': {'conteudo': {'query': q}}},
            '_source': ['id', 'titulo', 'tipo']}
    return client.search(index=INDEX_NAME, body=body)['hits']['hits']

def busca_knn_ollama(q: str, k: int = K):
    """kNN puro: embedding BGE-M3 vem do Ollama client-side."""
    q_emb = ollama_embed(q)
    body = {'size': k,
            'query': {'knn': {'knn_vector': {'vector': q_emb, 'k': k}}},
            '_source': ['id', 'titulo', 'tipo']}
    return client.search(index=INDEX_NAME, body=body)['hits']['hits']

def busca_hybrid_ollama(q: str, pipeline: str, k: int = K):
    """Hybrid: match (BM25) + knn com vetor BGE-M3 vindo do Ollama client-side,
    fundidos pelo search pipeline especificado.
    """
    q_emb = ollama_embed(q)
    body = {
        'size': k,
        'query': {
            'hybrid': {
                'queries': [
                    {'match': {'conteudo': {'query': q}}},
                    {'knn':   {'knn_vector': {'vector': q_emb, 'k': k}}},
                ]
            }
        },
        '_source': ['id', 'titulo', 'tipo']
    }
    return client.search(
        index=INDEX_NAME, body=body,
        params={'search_pipeline': pipeline}
    )['hits']['hits']

bm25_h   = busca_bm25(QUERY_TEXT)
knn_h    = busca_knn_ollama(QUERY_TEXT)
rrf_h    = busca_hybrid_ollama(QUERY_TEXT, pipeline_rrf)
minmax_h = busca_hybrid_ollama(QUERY_TEXT, pipeline_minmax)

def mostra(label, hits):
    print(f'\n{label}')
    for i, h in enumerate(hits, 1):
        print(f"  {i}. [{h['_score']:.4f}] {h['_source']['titulo'][:80]}")

mostra('BM25 (lexical)', bm25_h)
mostra('kNN (Ollama BGE-M3 client-side)', knn_h)
mostra('Hybrid RRF (BM25 + Ollama)', rrf_h)
mostra('Hybrid Min-Max (BM25 + Ollama)', minmax_h)

## 7. Tabela Comparativa

In [ ]:
def to_rows(hits, mode):
    rows = []
    for rank, h in enumerate(hits[:K], 1):
        t = h['_source']['titulo']
        rows.append({'Modo': mode, 'Rank': rank, 'Score': h['_score'],
                     'Titulo': (t[:60] + '…') if len(t) > 60 else t})
    return rows

df = pd.DataFrame(
    to_rows(bm25_h, 'BM25')   + to_rows(knn_h, 'kNN-Ollama') +
    to_rows(rrf_h,  'RRF')    + to_rows(minmax_h, 'Min-Max')
)
print(f'\nQuery: {QUERY_TEXT}\n')
print(df.to_string(index=False))

## 8. Visualização dos Scores

In [ ]:
modes   = ['BM25', 'kNN-Ollama', 'RRF', 'Min-Max']
scores  = {m: [h['_score'] for h in lst[:K]]
           for m, lst in zip(modes, [bm25_h, knn_h, rrf_h, minmax_h])}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(f'Scores — Query: {QUERY_TEXT}', fontsize=13, fontweight='bold')
for ax, (m, s) in zip(axes.flat, scores.items()):
    if s:
        x = np.arange(1, len(s) + 1)
        ax.bar(x, s, color='#4C72B0', alpha=0.85, edgecolor='black')
        ax.set_title(m, fontweight='bold')
        ax.set_xlabel('Rank'); ax.set_ylabel('Score')
        ax.set_xticks(x); ax.grid(axis='y', alpha=0.3)
        for xi, si in zip(x, s):
            ax.text(xi, si, f'{si:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.savefig('lab2_scores.png', dpi=140); plt.show()

## 9. Análise — Lexical vs Semântica

**Insight prático:** queries com **terminologia técnica exata** (números de leis, palavras-chave) tendem a se beneficiar mais de BM25. Queries em **linguagem natural / paráfrases** se beneficiam de kNN (BGE-M3, Ollama). **Hybrid + RRF** é o default seguro quando o tipo da query é desconhecido.

In [ ]:
lex = 'Lei 8.666/93 dispensa de licitação'
sem = 'contratação direta sem concorrência por excepcionalidade'

for q in [lex, sem]:
    print(f'\n=== "{q}" ===')
    for label, fn in [('BM25',          busca_bm25),
                       ('kNN-Ollama',    busca_knn_ollama),
                       ('Hybrid_RRF',    lambda x: busca_hybrid_ollama(x, pipeline_rrf))]:
        hits = fn(q)
        top = hits[0]['_source']['titulo'][:70] if hits else '(sem resultado)'
        score = hits[0]['_score'] if hits else 0
        print(f'  {label:12} | top: [{score:.3f}] {top}')

## 10. Latência do Embedding Ollama (Trade-off)

Como o embedding agora é gerado no cliente, vale medir o custo extra por query.

In [ ]:
queries_amostra = [
    'operação de crédito com garantia da União',
    'tomada de contas especial dano ao erário',
    'representação irregularidades em contratação',
    'recurso de reconsideração tribunal de contas',
    'auditoria operacional desempenho gestão pública',
]
rows = []
for q in queries_amostra:
    t0 = time.perf_counter(); ollama_embed(q); lat_emb = (time.perf_counter()-t0)*1000
    t0 = time.perf_counter(); busca_knn_ollama(q); lat_knn = (time.perf_counter()-t0)*1000
    t0 = time.perf_counter(); busca_bm25(q);        lat_b25 = (time.perf_counter()-t0)*1000
    rows.append({'query': q[:50], 'emb_ms': lat_emb,
                 'knn_total_ms': lat_knn, 'bm25_ms': lat_b25})
df_lat = pd.DataFrame(rows).round(1)
print(df_lat.to_string(index=False))
print(f"\nMédia — embedding Ollama: {df_lat['emb_ms'].mean():.1f} ms")
print(f"Média — kNN total (inclui embedding): {df_lat['knn_total_ms'].mean():.1f} ms")
print(f"Média — BM25 puro: {df_lat['bm25_ms'].mean():.1f} ms")

## 11. Exercício Proposto

Crie 3 variantes de pipelines com `rank_constant` diferentes (10, 60, 200) e meça MRR@5 em 5 queries do corpus TCU 2026 (use `queries_avaliacao_aula4.json`):

- `rank_constant=10`: prioriza top-ranked
- `rank_constant=60`: default (Cormack et al.)
- `rank_constant=200`: suaviza diferenças entre ranks

Qual valor maximiza MRR@5 no domínio jurídico? Por que?

## Referências (ABNT)

BOGAN, R.; GAIEVSKI, M.; SHAH, M.; KOLCHINA, F. **Introducing reciprocal rank fusion for hybrid search**. OpenSearch Blog, 12 fev. 2025. <https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/>.

CORMACK, G. V.; CLARKE, C. L. A.; BUETTCHER, S. **Reciprocal Rank Fusion outperforms Condorcet and individual rank learning methods**. SIGIR, 2009.

OPENSEARCH PROJECT. **Hybrid query / Search pipelines**. OpenSearch 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/hybrid-search/>.

CHEN, J. et al. **BGE M3-Embedding**. arXiv:2309.07597, 2024.

OLLAMA. **BGE-M3 model card**. <https://ollama.com/library/bge-m3>.